# MAIA AISF Hack S26 — Interpretability Starter Notebook

**Read this in Colab before touching RunPod.**

This notebook does three things:
1. Shows you the environment and imports you'll use
2. Runs a toy example so you understand the data structures
3. Walks you through the sharpening questions so you have a research question before you start

On RunPod this will be fast. On Colab CPU it will be slow — that's fine, you're here to read, not run experiments.

## Cell 1 — Install dependencies
Run this first. Takes ~60s on Colab, ~15s on RunPod.

In [ ]:
!pip install transformer_lens einops plotly pandas matplotlib -q

## Cell 2 — Load model
We use GPT-2 small as the default. It's well-studied, fast to load, and has good tooling support.
On Colab CPU this takes ~2 min. On RunPod RTX 4090 it takes ~8s.
Don't switch to a larger model without asking a mentor first.

In [ ]:
import torch
from transformer_lens import HookedTransformer
import einops
import plotly.express as px
import pandas as pd

model = HookedTransformer.from_pretrained('gpt2')
print(f'Loaded: {model.cfg.model_name}')
print(f'Layers: {model.cfg.n_layers}, Heads: {model.cfg.n_heads}, d_model: {model.cfg.d_model}')

## Cell 3 — Forward pass and activation cache
This is the core pattern you'll use in every experiment.
`run_with_cache` returns logits AND a cache of all intermediate activations.
The cache is how you access attention patterns, residual stream, MLP outputs, etc.

In [ ]:
prompt = 'The Eiffel Tower is in the city of'
tokens = model.to_tokens(prompt)
token_strs = model.to_str_tokens(prompt)

logits, cache = model.run_with_cache(tokens)

# What's in the cache?
print('Cache keys (first 10):')
for key in list(cache.keys())[:10]:
    print(f'  {key}: {cache[key].shape}')

## Cell 4 — Toy example: attention pattern visualization
Let's look at what head 0 in layer 0 attends to.
This is the pattern you'll use to investigate any attention head.

In [ ]:
# Attention pattern for layer 0, head 0
# Shape: [batch, head, dest_pos, src_pos]
attn_pattern = cache['pattern', 0]  # layer 0
head_0_pattern = attn_pattern[0, 0]  # batch 0, head 0

fig = px.imshow(
    head_0_pattern.detach().numpy(),
    labels=dict(x='Source', y='Destination', color='Attention'),
    x=token_strs,
    y=token_strs,
    title='Layer 0, Head 0 — Attention Pattern'
)
fig.show()

# Save to results/ — do this for every figure you want to keep
import os
os.makedirs('results', exist_ok=True)
fig.write_image('results/attn_L0H0.png')

## Cell 5 — YOUR RESEARCH QUESTION

Before you move to RunPod, answer these three questions in this cell.
Don't skip this — it's the most important 10 minutes of your day.

A bad answer: *'We want to study attention heads'*
A good answer: *'We want to know whether head 4.4 is responsible for indirect object identification because we saw it activate strongly on subject tokens in the readings'*

**Question 1: What are you investigating?**

We want to know whether _______ because _______

---

**Question 2: What's your positive vs negative example set?**

Positive examples (concept IS present): 

Negative examples (concept IS NOT present): 

---

**Question 3: What does a result look like?**

At demo time we will show: _______ (describe the plot or number)

If we find X it means _______

If we don't find X it means _______

## Cell 6 — Saving results
Run this pattern every time you produce a figure worth keeping.
Don't wait until the end — save as you go.

In [ ]:
import os
os.makedirs('results', exist_ok=True)

# Plotly figures
# fig.write_image('results/descriptive_name.png')

# Matplotlib figures
# plt.savefig('results/descriptive_name.png', dpi=150, bbox_inches='tight')

# DataFrames
# df.to_csv('results/descriptive_name.csv', index=False)

print('Results saved. Now move to scratch.ipynb on RunPod.')

## Ready to start

You've answered the sharpening questions. Now:

1. Request your RunPod key in `#aisf-hack-s26-compute-request`
2. Deploy your pod (RTX 4090, RunPod PyTorch 2.2)
3. Clone your team repo onto the pod
4. Open `scratch.ipynb` and start experimenting
5. Save everything to `results/` before your pod stops

Stuck? Post in `#aisf-hack-s26-mentor-queue`.